# 02 — CPU worker 1/4: 00002, 00003

        **Dừng phiên nhập tuần tự cũ → mỗi tài khoản mở đúng một worker → CPU → Run all.**
        Dùng chung folder VietMedBridge. Cả bốn worker chạy ngay, không cần chờ acc 0.
        WORKER_ID=1, TEAM_SIZE=4 đã điền sẵn; không đổi trong khi chạy.
        Không chạy hai phiên cùng worker ID. Luồng submission tiếp tục riêng.

        Nguồn hoàn tất dùng lại receipt. Nguồn dở tiếp tục checkpoint cũ;
        nguồn chưa chạy ghi riêng trong parallel_cpu4/jobs. Khi cả 4 worker xong,
        acc 0 chạy lại **cell 4** để kiểm đủ 9/9 nguồn và ID duy nhất.


In [ ]:
#@title 1. Kết nối Drive và cài code riêng cho lượt nhập mới
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Dùng runtime Colab Python 3.11–3.13.")
print("Bắt đầu CPU intake; không nạp model embedding.", flush=True)
from google.colab import drive
drive.mount("/content/drive")

# Chỉ sửa nếu project của team ở vị trí khác; tự tìm shortcut dùng chung.
PROJECT = Path("/content/drive/MyDrive/VietMedBridge")
if not (PROJECT / "data").is_dir():
    shared = Path("/content/drive/.shortcut-targets-by-id/1uKKHYEcMPodnJIscKPi6jsoNBJvyhFGK/VietMedBridge")
    if (shared / "data").is_dir():
        PROJECT = shared
    else:
        raise FileNotFoundError("Chưa tìm thấy folder VietMedBridge dùng chung. Thêm shortcut VietMedBridge vào My Drive.")
CAMPAIGN = "btc-new-71be0e12-20261019-v1"
INTAKE_ROOT = PROJECT.resolve() / "data_staging" / CAMPAIGN
INTAKE_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path("/content/vmb_intake_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
CHECKOUT = Path("/content/VietMedBridge-intake")
REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
lock_path = INTAKE_ROOT / "intake_code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.is_file() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("Intake code lock belongs to another repository.")
if not (CHECKOUT / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True).strip()
if remote != REPO_URL:
    raise ValueError("Checkout intake thuộc repo khác.")
reference = lock.get("commit") or "main"
print("Tải code intake:", reference, flush=True)
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
commit = subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True).strip()
if "vietmedbridge" in sys.modules and globals().get("_INTAKE_COMMIT") != commit:
    raise RuntimeError("Restart session để nạp đúng code intake; checkpoint trên Drive vẫn giữ.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", commit], check=True)
print("Cài dependencies CPU…", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(CHECKOUT)], check=True)
runtime_path = INTAKE_ROOT / "intake_runtime_lock.json"
if runtime_path.is_file():
    pinned = json.loads(runtime_path.read_text())
    packages = [name+"=="+version for name, version in pinned.items() if name != "vietmedbridge"]
    subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)
sys.path.insert(0, str(CHECKOUT / "src"))
importlib.invalidate_caches()
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/content/hf_cache_intake"
from vietmedbridge.artifacts import atomic_json, read_json, runtime_versions
from vietmedbridge.isolated_intake import intake_root
if INTAKE_ROOT != intake_root(PROJECT / "data", CAMPAIGN):
    raise ValueError("Intake output root không tách biệt khỏi submission.")
if not lock:
    atomic_json(lock_path, {"repo_url": REPO_URL, "commit": commit})
if not runtime_path.is_file():
    atomic_json(runtime_path, runtime_versions())
_INTAKE_COMMIT = commit
print("Code:", commit, "\nCheckpoint:", INTAKE_ROOT, flush=True)


In [ ]:
#@title 2. Nhận checkpoint; nối đủ 8 frontier và retained của BTC mới
RUN_MODE = "parallel" #@param ["sequential", "parallel", "collect"]
WORKER_ID = 1 #@param {type:"integer"}
TEAM_SIZE = 4
"""Manifest-first intake controller, embedded into the Colab notebook.

Kept outside src/ deliberately: changing resume orchestration must not change
the pinned producer fingerprint or invalidate existing document/chunk shards.
Completed receipts are NOT a fresh payload-integrity audit: only metadata is
read for status; mapping payloads are SHA-checked when actually used.
"""
from collections import Counter
from contextlib import contextmanager
from dataclasses import asdict
from pathlib import Path
import re
import tempfile

from vietmedbridge import isolated_intake as intake_core
from vietmedbridge.artifacts import (atomic_json, code_fingerprint, digest_json,
    read_json, runtime_versions, sha256_file, utc_now, verify_file)
from vietmedbridge.dataset import _connection


def intake_memory_mb():
    """At most 2 GiB and a quarter of available RAM; no reservation or GPU use."""
    available = []
    try:
        match = re.search(r"^MemAvailable:\s+(\d+) kB", Path("/proc/meminfo").read_text(), re.M)
        if match:
            available.append(int(match[1])*1024)
    except OSError:
        pass
    for limit, used in (("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
                        ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes")):
        try:
            available.append(max(0, int(Path(limit).read_text())-int(Path(used).read_text())))
        except (OSError, ValueError):
            pass
    return min(2048, max(64, min(available)//(4*1024**2))) if available else 512


def bounded_intake_parquet(con, query, destination, work_dir):
    """Flush wide text every 2,048 rows, then atomically publish the full file."""
    with tempfile.TemporaryDirectory(prefix="intake-table-", dir=work_dir) as temporary:
        path = Path(temporary)/"table.parquet"
        con.sql(query).write_parquet(str(path), compression="zstd", row_group_size=2048)
        return intake_core.publish_file(path, destination)


@contextmanager
def bounded_intake_sql(memory_mb=None):
    """Scoped I/O adapter for the single intake controller, including old locks.

    The pinned producer files, policies and completed selections stay untouched.
    Only this module's selection/outcome SQL and Parquet writer are adapted;
    normal importer/chunk shards still use their original producer.
    """
    budget = intake_memory_mb() if memory_mb is None else memory_mb
    if type(budget) is not int or not 64 <= budget <= 2048:
        raise ValueError("Intake SQL budget must be 64..2048 MiB.")
    original_connection, original_parquet = intake_core._connection, intake_core._parquet

    def connect(temp_dir, memory_limit=None):
        con = _connection(temp_dir, memory_limit=memory_limit or f"{budget}MiB")
        try:
            con.execute("SET threads=1")
            con.execute("SET preserve_insertion_order=false")
        except BaseException:
            con.close()
            raise
        return con

    intake_core._connection, intake_core._parquet = connect, bounded_intake_parquet
    print(f"Intake SQL: RAM limit {budget} MiB | 1 thread | local spill | Parquet groups 2,048 rows", flush=True)
    try:
        yield
    finally:
        intake_core._connection, intake_core._parquet = original_connection, original_parquet


class IntakeResume:
    def __init__(self, plan, *, root, work_dir, tokenizer_spec, chunk_config, shard_size=2048,
                 prior_mappings=()):
        self.plan, self.root, self.work = plan, Path(root), Path(work_dir)
        self.spec, self.config, self.shard_size = tokenizer_spec, chunk_config, shard_size
        self.plan_sha = digest_json(plan)
        self.prior_mappings = [(Path(path), checksum) for path, checksum in prior_mappings]
        if [checksum for _, checksum in self.prior_mappings] != plan.get("prior_mapping_sha256", []):
            raise ValueError("Prior intake mappings differ from the pinned continuation plan.")
        if any((p/"active_data_candidate.json").exists() for p in (self.root, *self.root.parents)):
            raise ValueError("Intake must stay outside the active submission data root.")
        if any(not re.fullmatch(r"[a-zA-Z0-9_-]+", s["job_id"]) for s in plan["sources"]):
            raise ValueError("Invalid intake job ID.")
        intake_core.pin_plan(self.root, plan)
        self.work.mkdir(parents=True, exist_ok=True)
        self.contract = {"tokenizer": tokenizer_spec, "chunking": asdict(chunk_config),
            "shard_size": shard_size, "code_sha256": code_fingerprint(), "runtime": runtime_versions()}
        contract = self.root/"intake_contract.json"
        if contract.exists() and read_json(contract) != self.contract:
            raise ValueError("Pinned intake code/runtime/chunk policy changed; do not discard checkpoints.")
        if not contract.exists():
            atomic_json(contract, self.contract)
        self.refresh()

    def _file(self, root, item):
        path = intake_core._inside(root, item["path"])
        stat = path.stat()  # Missing/changed files fail; never silently reprocess a completed source.
        if stat.st_size <= 0 or ("bytes" in item and stat.st_size != item["bytes"]):
            raise ValueError(f"Checkpoint artifact size changed: {path}")
        return path

    def refresh(self):
        self.completed = []
        gap = False
        for source in self.plan["sources"]:
            target = self.root/"sources"/source["job_id"]
            receipt = target/"complete.json"
            if not receipt.exists():
                gap = True
                continue
            if gap:
                raise ValueError("Out-of-order complete receipt; preserve files and inspect source lineage.")
            saved = read_json(receipt)
            if (saved.get("state") != "COMPLETE" or saved["source"] != source
                    or saved["plan_sha256"] != self.plan_sha):
                raise ValueError("Completed source/plan binding changed.")
            for kind, name, sha_key in (("added_ids", "added_ids.parquet", "added_ids_sha256"),
                                       ("outcomes", "source_outcomes.parquet", "outcomes_sha256")):
                item = saved.get("files", {}).get(kind, {"path": name, "sha256": saved[sha_key]})
                if item["sha256"] != saved[sha_key]:
                    raise ValueError("Receipt payload binding changed.")
                self._file(target, item)
            self.completed.append(saved)

    def status(self):
        counts = Counter()
        for saved in self.completed:
            counts.update(saved["counts"])
        return {"state": "INTAKE_COMPLETE" if len(self.completed) == len(self.plan["sources"]) else "INTAKE_IN_PROGRESS",
            "plan_sha256": self.plan_sha, "completed_sources": len(self.completed),
            "total_sources": len(self.plan["sources"]), "counts": dict(counts),
            "builds": [{"build_run": s["build_run"], "snapshot_sha256": s["build_snapshot_sha256"],
                "snapshot_id": s["source"]["snapshot_id"]} for s in self.completed if s["build_run"]],
            "resume_policy": "MANIFEST_FIRST_MAPPING_SHA_CHECKED_ON_USE",
            "scope": "CPU import, quality and exact-span chunks only; not embedded/frozen/promoted or submitted",
            "submission_candidate_changed": False, "updated_at": utc_now()}

    def _selection(self, source, target, existing):
        marker = target/"selection/selection.json"
        if not marker.exists():
            return None
        saved = read_json(marker)
        metadata = read_json(target/"metadata/metadata.json")
        expected = {"metadata": metadata,
            "links_sha256": self.plan["snapshots"][source["snapshot_id"]]["sha256"],
            "existing": [{"sha256": checksum} for _, checksum in existing]}
        if (saved["identity"] != expected or metadata["source"] != source
                or metadata["files"]["frontier"]["sha256"] != source["frontier_sha256"]
                or sum(saved["counts"].values()) != saved["official_ids"]):
            raise ValueError("Saved selection binding changed; no automatic rebuild.")
        for item in saved["files"].values():
            self._file(target/"selection", item)
        print(f"REUSE_SELECTION {source['job_id']} | no archive/baseline rescan", flush=True)
        return saved, metadata

    def _validated_build(self, source, selection, build_dir):
        path = build_dir/"build.json"
        if not path.exists():
            return None
        build = read_json(path)
        if not build.get("selected_range_complete"):
            return None
        if build.get("state") != "DATA_VALIDATED" or not build.get("integrity", {}).get("passed"):
            raise ValueError("Complete build has not passed integrity; retain and inspect it.")
        config = read_json(build_dir/"config.json")
        queued = selection["counts"].get("QUEUED_FOR_PIPELINE", 0)
        files = {kind: {"name": Path(intake_core.MEMBERS[kind]).name,
            "sha256": selection["files"][kind]["sha256"],
            "bytes": self._file(self.root/"sources"/source["job_id"]/"selection", selection["files"][kind]).stat().st_size}
            for kind in intake_core.MEMBERS}
        if (config["signature"] != digest_json({k: v for k, v in config.items() if k != "signature"})
                or config["signature"] != build["signature"]
                or config["source_input_signature"] != digest_json(files)
                or config["external_snapshot_id"] != source.get("original_snapshot_id", source["snapshot_id"])
                or config["official_links_sha256"] != self.plan["snapshots"][source["snapshot_id"]]["sha256"]
                or config["tokenizer"] != self.spec or config["chunks"] != asdict(self.config)
                or config["code_sha256"] != self.contract["code_sha256"]
                or config["runtime"] != self.contract["runtime"] or config["shard_size"] != self.shard_size
                or build["requested_input_records"] != queued or build["counts"]["input_records"] != queued
                or build["snapshot_sha256"] != digest_json({"signature": build["signature"], "parts": build["parts"]})):
            raise ValueError("Validated build identity differs from the saved selection/policy.")
        for part in build["parts"]:
            for kind, item in part["files"].items():
                path = self._file(build_dir, item)
                if kind in ("documents", "failures"):
                    # Only these payloads are about to be used to finish the
                    # ID/outcome mapping. Child/parent chunks stay deferred to freeze.
                    verify_file(path, item["sha256"])
        print(f"REUSE_VALIDATED_BUILD {source['job_id']} | no re-chunk or full validation replay", flush=True)
        return build

    @bounded_intake_sql()
    def run(self, *, load_reader, load_links, load_baseline, load_tokenizer, load_metadata=None,
            select_metadata=None, source_completed=None, max_new_sources=None):
        if max_new_sources is not None and (type(max_new_sources) is not int or max_new_sources < 1):
            raise ValueError("max_new_sources must be a positive integer.")
        newly_completed = 0
        self.refresh()
        for saved in self.completed:
            print(f"SKIP_COMPLETED_SOURCE {saved['source']['job_id']} | receipt only; payload checked on use", flush=True)
        if len(self.completed) == len(self.plan["sources"]):
            result = self.status()
            atomic_json(self.root/"intake_status.json", result)
            return result
        # Paths/checksums suffice until a NEW selection really needs a mapping.
        existing = [(None, self.plan["baseline"]["sha256"])] + self.prior_mappings + [
            (self.root/"sources"/s["source"]["job_id"]/"added_ids.parquet", s["added_ids_sha256"])
            for s in self.completed]
        for source in self.plan["sources"][len(self.completed):]:
            job = source["job_id"]
            target, build_dir = self.root/"sources"/job, self.root/"processed"/job
            print(f"CONTINUE_SOURCE {job} | {len(self.completed)}/{len(self.plan['sources'])} sources complete", flush=True)
            cached = self._selection(source, target, existing)
            if cached is None:
                metadata = (load_metadata(source, target/"metadata") if load_metadata is not None
                            else intake_core.stage_metadata(load_reader(), source, target/"metadata"))
                local_maps = [load_baseline() if path is None else intake_core._local_mapping(path, checksum, self.work)
                              for path, checksum in existing]
                for path, (_, checksum) in zip(local_maps, existing):
                    verify_file(path, checksum)
                links = load_links(source["snapshot_id"])
                selection = (select_metadata(metadata, links, local_maps, target/"selection") if select_metadata is not None
                    else intake_core.select_delta(target/"metadata", metadata, links, local_maps, target/"selection", self.work))
            else:
                selection, metadata = cached
            build = None
            if selection["counts"].get("QUEUED_FOR_PIPELINE", 0):
                build = self._validated_build(source, selection, build_dir)
                if build is None:
                    markers = sorted((build_dir/"parts").glob("*.done.json"))
                    committed = [read_json(path) for path in markers]
                    print(f"RESUME_PARTS {job} | {len(committed)} committed shards, "
                          f"{sum(p['records'] for p in committed):,} IDs; verify current source only", flush=True)
                    staged = intake_core.restore_import_source(target/"selection", selection, self.work/"sources"/job)
                    for kind in intake_core.MEMBERS:
                        item = selection["files"][kind]
                        verify_file(staged/item["path"], item["sha256"])
                    tokenizer, actual_spec = load_tokenizer()
                    if actual_spec != self.spec:
                        raise ValueError("Loaded tokenizer differs from the pinned contract.")
                    build = intake_core.import_external_corpus(staged, load_links(source["snapshot_id"]),
                        self.root/"processed", tokenizer, self.spec, run_name=job, config=self.config,
                        work_dir=self.work, shard_size=self.shard_size)
                    if not build["selected_range_complete"] or not build["integrity"]["passed"]:
                        raise ValueError("Intake build is not complete and valid.")
                atomic_json(build_dir/"intake_provenance.json", {"source": source,
                    "original_metadata": metadata["files"], "plan_sha256": self.plan_sha,
                    "membership_snapshot_id": source["snapshot_id"],
                    "original_snapshot_id": source.get("original_snapshot_id", source["snapshot_id"]),
                    "portable_origin": metadata.get("portable_origin"),
                    "raw_locator": "Drive file ID -> optional inner_tar -> suffix after # in document.raw_file",
                    "raw_verification": "METADATA_ONLY_RAW_PAYLOAD_NOT_REPLAYED"})
            entry = selection["files"]["selection"]
            verify_file(target/"selection"/entry["path"], entry["sha256"])
            counts = intake_core._write_outcomes(target, target/"selection", selection, build_dir, build, self.work)
            with _connection(self.work) as con:
                if build:
                    con.read_parquet([str(build_dir/p["files"]["documents"]["path"]) for p in build["parts"]]).create_view("added")
                    query = "SELECT doc_id,url FROM added"
                else:
                    query = "SELECT NULL::BIGINT AS doc_id,NULL::VARCHAR AS url WHERE false"
                intake_core._parquet(con, query, target/"added_ids.parquet", self.work)
            files = {kind: {"path": name, "sha256": sha256_file(target/name), "bytes": (target/name).stat().st_size}
                     for kind, name in (("added_ids", "added_ids.parquet"), ("outcomes", "source_outcomes.parquet"))}
            saved = {"source": source, "plan_sha256": self.plan_sha, "state": "COMPLETE", "counts": counts,
                "build_run": job if build else None, "build_snapshot_sha256": build["snapshot_sha256"] if build else None,
                "data_counts": build["counts"] if build else {}, "files": files,
                "added_ids_sha256": files["added_ids"]["sha256"], "outcomes_sha256": files["outcomes"]["sha256"],
                "updated_at": utc_now()}
            atomic_json(target/"complete.json", saved)  # last: durable commit of the whole source
            self.completed.append(saved)
            existing.append((target/"added_ids.parquet", saved["added_ids_sha256"]))
            result = self.status()
            atomic_json(self.root/"intake_status.json", result)
            print(f"SOURCE_CHECKPOINT_SAVED {job} | {counts}", flush=True)
            if source_completed is not None:
                source_completed()
            newly_completed += 1
            if max_new_sources is not None and newly_completed >= max_new_sources:
                return result
        return result

"""Portable retained adapter, embedded outside the pinned chunk producer.

Only audited TAR member ranges are downloaded. New BTC membership and original
crawl/extraction snapshots are distinct, checked identities throughout.
"""
import hashlib
import os
import tarfile
from pathlib import Path

from vietmedbridge import isolated_intake as intake_core
from vietmedbridge.artifacts import atomic_json, digest_json, read_json, sha256_file, verify_file


class RetainedAdapter:
    def __init__(self, spec, *, root, work_dir, parquet_writer):
        self.spec, self.root, self.work = spec, Path(root), Path(work_dir)
        self.write = parquet_writer
        self.root.mkdir(parents=True, exist_ok=True)
        self.work.mkdir(parents=True, exist_ok=True)

    def member(self, reader, kind):
        item = self.spec["members"][kind]
        path = intake_core._inside(self.root, item["path"])
        marker = path.with_suffix(path.suffix+".download.json")
        partial = path.with_suffix(path.suffix+".partial")
        identity = {"file_id": self.spec["drive_file_id"], "archive_bytes": self.spec["archive_bytes"], **item}
        saved = read_json(marker) if marker.exists() else None
        if saved and saved["identity"] != identity:
            raise ValueError("Retained member download identity changed.")
        if path.exists():
            if not saved or path.stat().st_size != item["bytes"]:
                raise ValueError("Unbound or truncated retained member; preserve it for inspection.")
            return path  # SHA checked by _local_mapping when used, never for status only.
        remote_meta = reader.metadata(self.spec["drive_file_id"])
        if int(remote_meta["size"]) != self.spec["archive_bytes"]:
            raise ValueError("Retained archive size changed.")
        if saved and saved["remote"] != remote_meta:
            raise ValueError("Retained archive changed during member download.")
        if not (512 <= item["offset"] and 0 < item["bytes"] <= 2*1024**3
                and item["offset"]+item["bytes"] <= self.spec["archive_bytes"]):
            raise ValueError("Retained member range is outside the audited archive.")
        offset = saved["completed_bytes"] if saved else 0
        if offset and (not partial.exists() or partial.stat().st_size < offset):
            raise ValueError("Retained download checkpoint is missing committed bytes.")
        path.parent.mkdir(parents=True, exist_ok=True)
        with reader.open(self.spec["drive_file_id"], remote_meta) as remote:
            remote.seek(item["offset"]-512)
            header = tarfile.TarInfo.frombuf(remote.read(512), "utf-8", "surrogateescape")
            if header.name != item["path"] or header.size != item["bytes"] or not header.isfile():
                raise ValueError("Retained member TAR header differs from the audited range.")
            # Bound each request, including the large documents member.
            if isinstance(remote, intake_core.DriveRange):
                remote.block_size = 8*1024**2
            with partial.open("r+b" if partial.exists() else "w+b") as outgoing:
                if offset:
                    outgoing.seek(saved["last_start"])
                    if hashlib.sha256(outgoing.read(offset-saved["last_start"])).hexdigest() != saved["last_sha256"]:
                        raise ValueError("Committed retained download block changed.")
                outgoing.truncate(offset)  # discard only this unpublished member's uncommitted tail
                outgoing.seek(offset)
                remote.seek(item["offset"]+offset)
                print(f"RETAINED_MEMBER {kind} | resume {offset:,}/{item['bytes']:,} bytes", flush=True)
                while offset < item["bytes"]:
                    start = offset
                    payload = remote.read(min(8*1024**2, item["bytes"]-offset))
                    if not payload:
                        raise EOFError("Truncated retained member range.")
                    outgoing.write(payload)
                    outgoing.flush()
                    offset += len(payload)
                    saved = {"identity": identity, "remote": remote_meta, "completed_bytes": offset,
                        "last_start": start, "last_sha256": hashlib.sha256(payload).hexdigest()}
                    atomic_json(marker, saved)
                    print(f"RETAINED_DOWNLOAD_CHECKPOINT {kind} | {offset:,}/{item['bytes']:,} bytes", flush=True)
        if reader.metadata(self.spec["drive_file_id"]) != remote_meta:
            raise ValueError("Retained archive changed during download; no member published.")
        verify_file(partial, item["sha256"])
        os.replace(partial, path)
        return path

    def local(self, reader, kind):
        return intake_core._local_mapping(self.member(reader, kind), self.spec["members"][kind]["sha256"], self.work)

    def prepare(self, reader):
        marker = self.root/"metadata.json"
        if marker.exists():
            saved = read_json(marker)
            if saved["portable_origin"] != self.spec:
                raise ValueError("Retained portable source changed; preserve existing checkpoints.")
            for kind in ("frontier", "dataset"):
                verify_file(self.root/saved["files"][kind]["path"], saved["files"][kind]["sha256"])
            return saved
        mapping, raw = self.local(reader, "mapping"), self.local(reader, "raw_catalog")
        descriptor = read_json(self.local(reader, "dataset"))
        if descriptor["snapshot_id"] != self.spec["snapshot_id"]:
            raise ValueError("Retained membership snapshot differs from BTC snapshot.")
        descriptor.update(snapshot_id=self.spec["original_snapshot_id"],
            membership_snapshot_id=self.spec["snapshot_id"],
            adapter="retained-portable-v1; original crawl/extraction snapshots preserved")
        with intake_core._connection(self.work) as con:
            con.read_parquet(str(mapping)).create_view("mapping")
            con.read_parquet(str(raw)).create_view("raw")
            name = intake_core.MEMBERS["frontier"]
            self.write(con, """SELECT m.crawl_url_id,r.fetch_url,count(*) AS doc_count
                FROM mapping m JOIN raw r USING(crawl_url_id)
                GROUP BY m.crawl_url_id,r.fetch_url ORDER BY m.crawl_url_id""", self.root/name, self.work)
        atomic_json(self.root/intake_core.MEMBERS["dataset"], descriptor)
        files = {kind: {"path": name, "sha256": sha256_file(self.root/name), "bytes": (self.root/name).stat().st_size}
                 for kind, name in (("frontier", intake_core.MEMBERS["frontier"]),
                                    ("dataset", intake_core.MEMBERS["dataset"]))}
        source = {k: v for k, v in self.spec.items() if k != "members"}
        source.update(frontier=files["frontier"]["path"], frontier_sha256=files["frontier"]["sha256"],
            portable_source_sha256=digest_json(self.spec))
        metadata = {"source": source, "files": files, "portable_origin": self.spec,
            "raw_verification": "PINNED_METADATA_ONLY_RAW_PAYLOAD_NOT_REPLAYED"}
        atomic_json(marker, metadata)
        return metadata

    def select(self, reader, metadata, links, existing_paths, output):
        output = Path(output)
        output.mkdir(parents=True, exist_ok=True)
        identity = {"metadata": metadata, "links_sha256": sha256_file(links),
                    "existing": [{"sha256": sha256_file(p)} for p in existing_paths]}
        expected = self.spec["official_links_sha256"]
        if identity["links_sha256"] != expected:
            raise ValueError("Retained BTC links checksum differs from the audited package.")
        mapping, raw = self.local(reader, "mapping"), self.local(reader, "raw_catalog")
        with intake_core._connection(self.work) as con:
            for view, path in (("mapping", mapping), ("raw", raw), ("official", links)):
                con.read_parquet(str(path)).create_view(view)
            con.read_parquet([str(p) for p in existing_paths], union_by_name=True).create_view("existing")
            checks = {}
            checks["duplicate_mapping_id"] = con.sql("SELECT count(*)-count(DISTINCT doc_id) FROM mapping").fetchone()[0]
            checks["duplicate_raw_id"] = con.sql("SELECT count(*)-count(DISTINCT crawl_url_id) FROM raw").fetchone()[0]
            queries = {
                "unmapped_native_id": "SELECT count(*) FROM mapping m ANTI JOIN official o ON m.doc_id=o.id AND m.original_url=o.url",
                "mapping_missing_raw": "SELECT count(*) FROM mapping m ANTI JOIN raw r USING(crawl_url_id)",
                "raw_outside_mapping": "SELECT count(*) FROM raw r ANTI JOIN mapping m USING(crawl_url_id)",
                "native_url_conflict": "SELECT count(*) FROM mapping m JOIN existing e USING(doc_id) WHERE m.original_url IS DISTINCT FROM e.url",
                "fetch_url_mismatch": "SELECT count(*) FROM mapping m JOIN raw r USING(crawl_url_id) WHERE split_part(trim(m.original_url),'#',1) IS DISTINCT FROM r.fetch_url",
                "invalid_raw_locator": "SELECT count(*) FROM raw WHERE raw_path IS NULL OR NOT starts_with(raw_path,'raw/') OR contains(raw_path,'..') OR contains(raw_path,chr(92))",
                "invalid_raw_metadata": "SELECT count(*) FROM raw WHERE raw_sha256 IS NULL OR NOT regexp_full_match(raw_sha256,'[0-9a-f]{64}') OR raw_size_bytes IS NULL OR raw_size_bytes<=0",
                "invalid_original_crawl": """SELECT count(*) FROM raw WHERE NOT json_valid(original_crawl_record_json)
                    OR json_extract_string(original_crawl_record_json,'$.status') IS DISTINCT FROM 'SUCCESS_RAW'
                    OR json_extract_string(original_crawl_record_json,'$.snapshot_id') IS DISTINCT FROM original_snapshot_id
                    OR json_extract_string(original_crawl_record_json,'$.raw_sha256') IS DISTINCT FROM raw_sha256
                    OR json_extract_string(original_crawl_record_json,'$.fetch_url') IS DISTINCT FROM fetch_url""",
            }
            for name, query in queries.items():
                checks[name] = con.sql(query).fetchone()[0]
            checks["snapshot_mismatch"] = con.execute("""SELECT count(*) FROM raw WHERE
                new_snapshot_id IS DISTINCT FROM ? OR original_snapshot_id IS DISTINCT FROM ?""",
                [self.spec["snapshot_id"], self.spec["original_snapshot_id"]]).fetchone()[0]
            con.execute("CREATE TABLE mapped AS SELECT crawl_url_id,doc_id,original_url AS url FROM mapping")
            checks["missing_official_alias"] = con.sql("""SELECT count(*) FROM official o JOIN raw r
                ON split_part(trim(o.url),'#',1)=r.fetch_url ANTI JOIN mapped m ON m.doc_id=o.id AND m.url=o.url""").fetchone()[0]
            self._audit(output, checks)
            con.execute("""CREATE TABLE selection AS SELECT m.*, CASE WHEN NOT EXISTS (
                SELECT 1 FROM mapped a WHERE a.crawl_url_id=m.crawl_url_id AND NOT EXISTS (
                    SELECT 1 FROM existing e WHERE e.doc_id=a.doc_id AND e.url=a.url))
                THEN 'REUSED_EXISTING_DOCUMENT' ELSE 'QUEUED_FOR_PIPELINE' END AS action FROM mapped m""")
            con.execute("CREATE VIEW needed AS SELECT DISTINCT crawl_url_id FROM selection WHERE action='QUEUED_FOR_PIPELINE'")
            counts = dict(con.sql("SELECT action,count(*) FROM selection GROUP BY 1").fetchall())
            print(f"RETAINED_SELECTION | {counts} native IDs | original snapshot retained", flush=True)
            if counts.get("QUEUED_FOR_PIPELINE", 0):
                documents = self.local(reader, "documents")
                con.read_parquet(str(documents)).create_view("portable_documents")
                checks["duplicate_extraction"] = con.sql("SELECT count(*)-count(DISTINCT crawl_url_id) FROM portable_documents").fetchone()[0]
                checks["extraction_binding"] = con.sql("""SELECT count(*) FROM portable_documents e LEFT JOIN raw r USING(crawl_url_id)
                    WHERE r.crawl_url_id IS NULL OR e.snapshot_id IS DISTINCT FROM r.original_snapshot_id
                    OR e.raw_sha256 IS DISTINCT FROM r.raw_sha256 OR e.extract_status IS DISTINCT FROM r.extract_status
                    OR e.extract_status IS DISTINCT FROM 'EXTRACT_SUCCESS'""").fetchone()[0]
                checks["missing_successful_extraction"] = con.sql("""SELECT count(*) FROM raw r ANTI JOIN
                    portable_documents e USING(crawl_url_id) WHERE r.extract_status='EXTRACT_SUCCESS'""").fetchone()[0]
                checks["invalid_source_length"] = con.sql("""SELECT count(*) FROM portable_documents WHERE text IS NULL
                    OR length(trim(text))=0 OR char_count IS DISTINCT FROM length(text)""").fetchone()[0]
                self._audit(output, checks)
                con.execute("""CREATE VIEW normalized_extracted AS SELECT r.crawl_url_id,r.raw_sha256,
                    r.original_snapshot_id AS snapshot_id,
                    CASE WHEN e.extract_status='EXTRACT_SUCCESS' AND (
                        upper(coalesce(e.quality_tier,'')) IN ('LOW','QUARANTINE') OR
                        upper(coalesce(r.quality_tier,'')) IN ('LOW','QUARANTINE'))
                        THEN 'RETAINED_QUALITY_REVIEW_'||CASE
                            WHEN upper(coalesce(e.quality_tier,'')) IN ('LOW','QUARANTINE')
                            THEN upper(e.quality_tier) ELSE upper(r.quality_tier) END
                        ELSE coalesce(e.extract_status,r.extract_status,'MISSING_EXTRACTION') END AS extract_status,
                    e.text,e.title,e.char_count,e.text_markdown,e.language,e.language_method,e.language_confidence,
                    e.extractor,e.extractor_version,e.source_issue FROM raw r JOIN needed n USING(crawl_url_id)
                    LEFT JOIN portable_documents e USING(crawl_url_id)""")
            else:
                con.execute("""CREATE VIEW normalized_extracted AS SELECT NULL::VARCHAR crawl_url_id,
                    NULL::VARCHAR raw_sha256,NULL::VARCHAR snapshot_id,NULL::VARCHAR extract_status,
                    NULL::VARCHAR text,NULL::VARCHAR title,NULL::BIGINT char_count,NULL::VARCHAR text_markdown,
                    NULL::VARCHAR language,NULL::VARCHAR language_method,NULL::DOUBLE language_confidence,
                    NULL::VARCHAR extractor,NULL::VARCHAR extractor_version,NULL::VARCHAR source_issue WHERE false""")
            con.read_parquet(str(self.root/metadata["files"]["frontier"]["path"])).create_view("frontier")
            queries = {
                "frontier": "SELECT f.* FROM frontier f JOIN needed n USING(crawl_url_id) ORDER BY f.crawl_url_id",
                "crawl": """SELECT r.crawl_url_id,r.fetch_url,'SUCCESS_RAW' AS status,r.final_url,r.content_type,
                    r.raw_path,r.raw_sha256,r.raw_size_bytes,
                    try_cast(json_extract_string(r.original_crawl_record_json,'$.elapsed_ms') AS DOUBLE) AS elapsed_ms,
                    try_cast(json_extract_string(r.original_crawl_record_json,'$.http_status') AS INTEGER) AS http_status,
                    r.crawl_timestamp,r.original_snapshot_id AS snapshot_id,r.new_snapshot_id
                    FROM raw r JOIN needed n USING(crawl_url_id) ORDER BY r.crawl_url_id""",
                "extracted": "SELECT * FROM normalized_extracted ORDER BY crawl_url_id",
            }
            files = {}
            for kind, query in queries.items():
                name = intake_core.MEMBERS[kind]
                checksum = self.write(con, query, output/name, self.work)
                files[kind] = {"path": name, "sha256": checksum}
            name = intake_core.MEMBERS["dataset"]
            files["dataset"] = {"path": name, "sha256": intake_core.publish_file(self.root/metadata["files"]["dataset"]["path"], output/name)}
            checksum = self.write(con, "SELECT * FROM selection ORDER BY doc_id", output/"source_selection.parquet", self.work)
            files["selection"] = {"path": "source_selection.parquet", "sha256": checksum}
            result = {"identity": identity, "counts": counts, "official_ids": sum(counts.values()), "files": files}
            atomic_json(output/"selection.json", result)
            return result

    def _audit(self, output, checks):
        report = {"passed": not any(checks.values()), "checks": checks,
            "extraction_payload_checked": "extraction_binding" in checks,
            "membership_snapshot_id": self.spec["snapshot_id"], "original_snapshot_id": self.spec["original_snapshot_id"],
            "raw_verification": "PINNED_METADATA_BINDINGS_ONLY_RAW_PAYLOAD_NOT_REPLAYED"}
        atomic_json(output/"retained_input_audit.json", report)
        if not report["passed"]:
            raise ValueError(f"Retained integrity failed: {checks}")

"""Run the complete audited folder selection without rewriting the original plan."""
from collections import Counter
from pathlib import Path

from vietmedbridge.artifacts import atomic_json, digest_json, read_json, utc_now


class FullIntake:
    def __init__(self, scope, *, root, work_dir, tokenizer_spec, chunk_config,
                 controller_type, retained_type, sql_context, parquet_writer, shard_size=2048):
        self.scope, self.root, self.work = scope, Path(root), Path(work_dir)
        if any((p/"active_data_candidate.json").exists() for p in (self.root, *self.root.parents)):
            raise ValueError("Full intake must stay outside the active submission data root.")
        self.controller_type, self.retained_type = controller_type, retained_type
        self.sql_context, self.parquet_writer = sql_context, parquet_writer
        self.options = dict(work_dir=self.work, tokenizer_spec=tokenizer_spec,
                            chunk_config=chunk_config, shard_size=shard_size)
        base = scope["base_plan"]
        required = base["required_snapshot_id"]
        sources = base["sources"]+scope["additional_sources"]+[scope["retained"]]
        if (scope["base_plan_sha256"] != digest_json(base) or set(base["snapshots"]) != {required}
                or not sources or any(s["snapshot_id"] != required or s["mode"] != "NEW_SNAPSHOT_SOURCE" for s in sources)
                or len({s["job_id"] for s in sources}) != len(sources)):
            raise ValueError("Full intake must contain only audited new BTC membership sources.")
        self.root.mkdir(parents=True, exist_ok=True)
        marker = self.root/"full_intake_scope.json"
        if marker.exists() and read_json(marker) != scope:
            raise ValueError("Audited folder scope changed; preserve checkpoints and audit the additions.")
        if not marker.exists():
            atomic_json(marker, scope)
        self.base = controller_type(base, root=self.root, **self.options)
        self.frontiers = self.retained = None
        self.extension_root = self.root/"additional-frontiers"
        self.retained_root = self.root/"retained-recovery"
        if (self.extension_root/"intake_plan.json").exists():
            if len(self.base.completed) != len(base["sources"]):
                raise ValueError("Continuation exists before all base sources completed.")
            prior = self._mappings(self.base)
            plan = self._plan(scope["additional_sources"], prior)
            self.frontiers = controller_type(plan, root=self.extension_root, prior_mappings=prior, **self.options)
        if (self.retained_root/"intake_plan.json").exists():
            if self.frontiers is None or len(self.frontiers.completed) != len(scope["additional_sources"]):
                raise ValueError("Retained continuation exists before the additional frontiers completed.")
            prior = self._mappings(self.base, self.frontiers)
            plan = read_json(self.retained_root/"intake_plan.json")
            if (len(plan["sources"]) != 1 or plan["sources"][0].get("portable_source_sha256") != digest_json(scope["retained"])
                    or plan != self._plan(plan["sources"], prior)):
                raise ValueError("Retained continuation source/dependencies changed.")
            self.retained = controller_type(plan, root=self.retained_root, prior_mappings=prior, **self.options)

    def _mappings(self, *controllers):
        return [(controller.root/"sources"/saved["source"]["job_id"]/"added_ids.parquet", saved["added_ids_sha256"])
                for controller in controllers for saved in controller.completed]

    def _plan(self, sources, prior):
        base = self.scope["base_plan"]
        return {**base, "campaign": "continuation-"+digest_json(sources)[:16], "sources": sources,
            "parent_plan_sha256": self.scope["base_plan_sha256"],
            "full_scope_sha256": digest_json(self.scope),
            "prior_mapping_sha256": [checksum for _, checksum in prior]}

    def _status(self):
        controllers = [self.base]
        if self.frontiers is not None:
            controllers.append(self.frontiers)
        if self.retained is not None:
            controllers.append(self.retained)
        counts, builds, completed_jobs = Counter(), [], []
        for controller in controllers:
            controller.refresh()
            phase = controller.status()
            counts.update(phase["counts"])
            builds.extend({**build, "phase_root": str(controller.root.relative_to(self.root))} for build in phase["builds"])
            completed_jobs.extend(s["source"]["job_id"] for s in controller.completed)
        jobs = [s["job_id"] for s in self.scope["base_plan"]["sources"]+self.scope["additional_sources"]+[self.scope["retained"]]]
        result = {"state": "FULL_INTAKE_COMPLETE" if set(completed_jobs) == set(jobs) else "FULL_INTAKE_IN_PROGRESS",
            "completed_sources": len(completed_jobs), "total_sources": len(jobs), "counts": dict(counts),
            "pending_sources": [job for job in jobs if job not in completed_jobs], "builds": builds,
            "scope_sha256": digest_json(self.scope), "audited_coverage": self.scope["audited_coverage"],
            "submission_candidate_changed": False, "updated_at": utc_now(),
            "scope": "CPU import/quality/chunks for all audited new BTC sources; not embedding, index, or submission",
            "counts_unit": "native ID outcomes; ADDED/REUSED/FAILURE are distinct, not all successful documents"}
        atomic_json(self.root/"full_intake_status.json", result)
        print(f"FULL_INTAKE_PROGRESS {len(completed_jobs)}/{len(jobs)} sources complete", flush=True)
        return result

    def status(self):
        self._serial_guard()
        return self._status()

    def _serial_guard(self):
        if (self.root/"parallel_cpu4/parallel_plan.json").exists():
            raise ValueError("Lượt này đã chuyển sang 4 CPU workers. Mở notebook worker 0–3 hoặc chọn RUN_MODE='collect'; không chạy lại sequential.")

    def run(self, **loaders):
        self._serial_guard()
        print("FULL_INTAKE: original frontiers -> additional frontiers -> retained recovery", flush=True)
        self.base.run(**loaders, source_completed=self._status)
        self._status()
        prior = self._mappings(self.base)
        plan = self._plan(self.scope["additional_sources"], prior)
        self.frontiers = self.controller_type(plan, root=self.extension_root, prior_mappings=prior, **self.options)
        self.frontiers.run(**loaders, source_completed=self._status)
        self._status()
        prior = self._mappings(self.base, self.frontiers)
        adapter = self.retained_type(self.scope["retained"],
            root=self.retained_root/"sources"/self.scope["retained"]["job_id"]/"metadata",
            work_dir=self.work, parquet_writer=self.parquet_writer)
        marker = self.retained_root/"intake_plan.json"
        if marker.exists():
            plan = read_json(marker)
            sources = plan["sources"]
            if (len(sources) != 1 or sources[0].get("portable_source_sha256") != digest_json(self.scope["retained"])
                    or plan != self._plan(sources, prior)):
                raise ValueError("Retained continuation source/dependencies changed.")
        else:
            with self.sql_context():
                metadata = adapter.prepare(loaders["load_reader"]())
            plan = self._plan([metadata["source"]], prior)
        self.retained = self.controller_type(plan, root=self.retained_root, prior_mappings=prior, **self.options)

        def metadata_loader(source, target):
            if Path(target) != adapter.root:
                raise ValueError("Unexpected retained metadata directory.")
            metadata = adapter.prepare(loaders["load_reader"]())
            if metadata["source"] != source:
                raise ValueError("Retained conversion identity changed.")
            return metadata

        def selector(metadata, links, maps, output):
            return adapter.select(loaders["load_reader"](), metadata, links, maps, output)

        self.retained.run(**loaders, load_metadata=metadata_loader, select_metadata=selector,
                          source_completed=self._status)
        return self._status()

"""Four independent CPU workers with immutable source ownership and a reducer.

Legacy COMPLETE sources are references, not copied payloads. At most the one
legacy in-progress source is resumed in place; all untouched sources get their
own new controller root. Stop the serial notebook before starting these workers.
"""
from collections import Counter
from dataclasses import asdict
from pathlib import Path

from vietmedbridge import isolated_intake as intake_core
from vietmedbridge.artifacts import (atomic_json, code_fingerprint, digest_json,
    read_json, runtime_versions, sha256_file, utc_now, verify_file)


class ParallelIntake:
    OWNERS = (0, 0, 1, 1, 2, 0, 2, 3, 3)

    def __init__(self, scope, *, root, work_dir, tokenizer_spec, chunk_config,
                 controller_type, retained_type, sql_context, parquet_writer,
                 worker_id=0, team_size=4, shard_size=2048):
        if type(team_size) is not int or team_size != 4 or type(worker_id) is not int or not 0 <= worker_id < 4:
            raise ValueError("Dùng đúng TEAM_SIZE=4 và WORKER_ID=0,1,2,3; mỗi ID một phiên.")
        self.scope, self.root, self.work = scope, Path(root), Path(work_dir)
        if any((p/"active_data_candidate.json").exists() for p in (self.root, *self.root.parents)):
            raise ValueError("Parallel intake must stay outside the active submission data root.")
        self.sources = scope["base_plan"]["sources"]+scope["additional_sources"]+[scope["retained"]]
        required = scope["base_plan"]["required_snapshot_id"]
        if (not 1 <= len(self.sources) <= 9 or len({s["job_id"] for s in self.sources}) != len(self.sources)
                or any(s["snapshot_id"] != required or s["mode"] != "NEW_SNAPSHOT_SOURCE" for s in self.sources)
                or digest_json(scope["base_plan"]) != scope["base_plan_sha256"]):
            raise ValueError("Parallel sources must match the audited new BTC scope.")
        self.worker_id, self.parallel = worker_id, self.root/"parallel_cpu4"
        self.controller_type, self.retained_type = controller_type, retained_type
        self.sql_context, self.parquet_writer = sql_context, parquet_writer
        self.options = dict(work_dir=self.work, tokenizer_spec=tokenizer_spec,
                            chunk_config=chunk_config, shard_size=shard_size)
        self.owners = {s["job_id"]: self.OWNERS[i] for i, s in enumerate(self.sources)}
        self.contract = {"schema_version": 1, "scope_sha256": digest_json(scope), "team_size": 4,
            "owners": self.owners, "baseline_sha256": scope["base_plan"]["baseline"]["sha256"],
            "tokenizer": tokenizer_spec, "chunking": asdict(chunk_config), "shard_size": shard_size,
            "code_sha256": code_fingerprint(), "runtime": runtime_versions(),
            "dedup_policy": "FROZEN_BASELINE_PER_SOURCE_CROSS_SOURCE_IDS_CHECKED_AT_COLLECT"}
        marker = self.parallel/"parallel_plan.json"
        if marker.exists() and read_json(marker) != self.contract:
            raise ValueError("Parallel scope/team/runtime changed; do not overwrite existing checkpoints.")
        if not marker.exists():
            atomic_json(marker, self.contract)  # identical immutable content on all four workers
        self.plan_sha = digest_json(self.contract)

    def _path(self, relative):
        return intake_core._inside(self.root, relative)

    def _old_location(self, job):
        for root in (self.root, self.root/"additional-frontiers", self.root/"retained-recovery"):
            marker = root/"intake_plan.json"
            if marker.exists():
                plan = read_json(marker)
                source = next((s for s in plan["sources"] if s["job_id"] == job), None)
                if source:
                    self._check_plan(plan)
                    expected = next(s for s in self.sources if s["job_id"] == job)
                    self._check_source(source, expected)
                    return root, plan, source
        return None

    def _check_source(self, actual, expected):
        if expected.get("format") == "RETAINED_PORTABLE_V1":
            if (actual.get("portable_source_sha256") != digest_json(expected)
                    or any(actual.get(k) != v for k, v in expected.items() if k != "members")):
                raise ValueError("Retained source differs from the audited portable source.")
        elif actual != expected:
            raise ValueError("Source differs from the audited frontier.")

    def _check_plan(self, plan):
        base = self.scope["base_plan"]
        if (plan["baseline"] != base["baseline"] or plan["snapshots"] != base["snapshots"]
                or plan.get("required_snapshot_id") != base["required_snapshot_id"]):
            raise ValueError("Controller corpus/baseline differs from the pinned parallel scope.")
        for source in plan["sources"]:
            expected = next((s for s in self.sources if s["job_id"] == source["job_id"]), None)
            if expected is None:
                raise ValueError("Controller contains an unaudited source.")
            self._check_source(source, expected)

    def _receipt(self, root, plan, source):
        marker = root/"sources"/source["job_id"]/"complete.json"
        saved = read_json(marker)
        if saved["state"] != "COMPLETE" or saved["source"] != source or saved["plan_sha256"] != digest_json(plan):
            raise ValueError("Completed source binding changed.")
        for kind, name, sha_key in (("added_ids", "added_ids.parquet", "added_ids_sha256"),
                                   ("outcomes", "source_outcomes.parquet", "outcomes_sha256")):
            item = saved.get("files", {}).get(kind, {"path": name, "sha256": saved[sha_key]})
            path = intake_core._inside(marker.parent, item["path"])
            if (item["sha256"] != saved[sha_key] or not path.is_file() or path.stat().st_size <= 0
                    or ("bytes" in item and path.stat().st_size != item["bytes"])):
                raise ValueError("Completed source artifact is missing or changed.")
        return saved, marker

    def _ref(self, job):
        marker = self.parallel/"results"/(job+".json")
        if not marker.exists():
            return None
        ref = read_json(marker)
        if (ref["parallel_plan_sha256"] != self.plan_sha or ref["job_id"] != job
                or ref["worker_id"] != self.owners[job]):
            raise ValueError("Parallel result owner/scope changed.")
        root = self._path(ref["controller_root"])
        plan = read_json(root/"intake_plan.json")
        self._check_plan(plan)
        source = next(s for s in plan["sources"] if s["job_id"] == job)
        saved, receipt = self._receipt(root, plan, source)
        if sha256_file(receipt) != ref["receipt_sha256"]:
            raise ValueError("Committed parallel receipt changed.")
        return ref, saved, receipt

    def status(self):
        jobs = [job for job, owner in self.owners.items() if owner == self.worker_id]
        refs = [self._ref(job) for job in jobs]
        completed = [item[0]["job_id"] for item in refs if item is not None]
        return {"state": "CPU_WORKER_COMPLETE" if len(completed) == len(jobs) else "CPU_WORKER_PENDING",
            "worker_id": self.worker_id, "team_size": 4, "assigned_sources": jobs,
            "completed_sources": len(completed), "completed_jobs": completed,
            "total_sources": len(jobs), "pending_sources": [job for job in jobs if job not in completed],
            "parallel_plan_sha256": self.plan_sha, "submission_candidate_changed": False}

    def _dependencies(self, root, plan):
        expected = plan.get("prior_mapping_sha256", [])
        if not expected:
            return []
        refs = []
        for previous in (self.root, self.root/"additional-frontiers"):
            if previous == root or not (previous/"intake_plan.json").exists():
                break
            prior = read_json(previous/"intake_plan.json")
            self._check_plan(prior)
            for source in prior["sources"]:
                saved, receipt = self._receipt(previous, prior, source)
                refs.append((receipt.parent/"added_ids.parquet", saved["added_ids_sha256"]))
        if [checksum for _, checksum in refs] != expected:
            raise ValueError("Legacy in-progress source dependency mappings changed.")
        return refs

    def _bind(self, source):
        job = source["job_id"]
        marker = self.parallel/"bindings"/(job+".json")
        if marker.exists():
            saved = read_json(marker)
            if saved["parallel_plan_sha256"] != self.plan_sha or saved["worker_id"] != self.worker_id:
                raise ValueError("Job already belongs to another parallel worker or scope.")
            return saved
        old = self._old_location(job)
        kind, root = "NEW_SOURCE", self.parallel/"jobs"/job
        if old:
            old_root, _, _ = old
            if ((old_root/"sources"/job).exists() or (old_root/"processed"/job).exists()):
                kind, root = "LEGACY_SOURCE", old_root
        saved = {"parallel_plan_sha256": self.plan_sha, "worker_id": self.worker_id,
                 "job_id": job, "kind": kind, "controller_root": str(root.relative_to(self.root))}
        atomic_json(marker, saved)  # only the statically assigned owner writes this job
        return saved

    def _adapter(self, source, root):
        return self.retained_type(source, root=root/"sources"/source["job_id"]/"metadata",
            work_dir=self.work, parquet_writer=self.parquet_writer)

    def run(self, **loaders):
        print(f"CPU worker {self.worker_id}/4 | {self.status()['assigned_sources']}", flush=True)
        for source in self.sources:
            job = source["job_id"]
            if self.owners[job] != self.worker_id:
                continue
            if self._ref(job) is not None:
                print(f"SKIP_PARALLEL_COMPLETE {job} | receipt only", flush=True)
                continue
            binding = self._bind(source)
            root = self._path(binding["controller_root"])
            if binding["kind"] == "LEGACY_SOURCE":
                plan = read_json(root/"intake_plan.json")
                self._check_plan(plan)
                actual = next(s for s in plan["sources"] if s["job_id"] == job)
                prior = self._dependencies(root, plan)
            else:
                prior = []
                marker = root/"intake_plan.json"
                if marker.exists():
                    plan = read_json(marker)
                    self._check_plan(plan)
                    actual = plan["sources"][0]
                else:
                    actual = source
                    if source.get("format") == "RETAINED_PORTABLE_V1":
                        with self.sql_context():
                            actual = self._adapter(source, root).prepare(loaders["load_reader"]())["source"]
                    plan = {**self.scope["base_plan"], "sources": [actual], "campaign": "parallel-"+job,
                        "parallel_plan_sha256": self.plan_sha, "prior_mapping_sha256": []}
            receipt = root/"sources"/job/"complete.json"
            if not receipt.exists():
                controller = self.controller_type(plan, root=root, prior_mappings=prior, **self.options)
                pending = [s for s in plan["sources"] if s["job_id"] not in {v["source"]["job_id"] for v in controller.completed}]
                if not pending or pending[0]["job_id"] != job:
                    raise ValueError("Legacy resume is not this worker's source; stop serial intake before using parallel workers.")
                options = {}
                if source.get("format") == "RETAINED_PORTABLE_V1":
                    adapter = self._adapter(source, root)
                    options = {"load_metadata": lambda s, target: adapter.prepare(loaders["load_reader"]()),
                               "select_metadata": lambda m, links, maps, out: adapter.select(loaders["load_reader"](), m, links, maps, out)}
                controller.run(**loaders, **options, max_new_sources=1)
            saved, receipt = self._receipt(root, plan, actual)
            ref = {"parallel_plan_sha256": self.plan_sha, "job_id": job, "worker_id": self.worker_id,
                "controller_root": str(root.relative_to(self.root)), "receipt_sha256": sha256_file(receipt)}
            atomic_json(self.parallel/"results"/(job+".json"), ref)
            status = self.status()
            atomic_json(self.parallel/"workers"/(f"worker-{self.worker_id}.json"), status)
            print(f"PARALLEL_SOURCE_SAVED worker={self.worker_id} {job} | {saved['counts']}", flush=True)
        status = self.status()
        atomic_json(self.parallel/"workers"/(f"worker-{self.worker_id}.json"), status)
        return status

    def collect(self):
        refs = [self._ref(source["job_id"]) for source in self.sources]
        pending = [s["job_id"] for s, ref in zip(self.sources, refs) if ref is None]
        if pending:
            return {"state": "WAITING_FOR_CPU_WORKERS", "completed_sources": len(refs)-len(pending),
                    "total_sources": len(refs), "pending_sources": pending}
        cache = self.parallel/"collection.json"
        identity = {"parallel_plan_sha256": self.plan_sha,
                    "receipts": [ref[0] for ref in refs]}
        if cache.exists():
            saved = read_json(cache)
            if saved["identity"] != identity:
                raise ValueError("Collected source receipts changed; preserve the previous collection.")
            print("REUSED_PARALLEL_COLLECTION | no payload replay", flush=True)
            return saved["result"]
        counts, builds, paths = Counter(), [], []
        for ref, receipt, marker in refs:
            for kind, name, sha_key in (("added_ids", "added_ids.parquet", "added_ids_sha256"),
                                       ("outcomes", "source_outcomes.parquet", "outcomes_sha256")):
                item = receipt.get("files", {}).get(kind, {"path": name, "sha256": receipt[sha_key]})
                path = intake_core._inside(marker.parent, item["path"])
                verify_file(path, item["sha256"])
                if kind == "outcomes":
                    paths.append(intake_core._local_mapping(path, item["sha256"], self.work))
            counts.update(receipt["counts"])
            if receipt["build_run"]:
                builds.append({"build_run": receipt["build_run"], "snapshot_sha256": receipt["build_snapshot_sha256"],
                    "snapshot_id": receipt["source"]["snapshot_id"], "phase_root": ref["controller_root"]})
        with self.sql_context():
            with intake_core._connection(self.work) as con:
                con.read_parquet([str(p) for p in paths], union_by_name=True).create_view("outcomes")
                rows, unique_ids, unfinished = con.sql("""SELECT count(*),count(DISTINCT doc_id),
                    count(*) FILTER (WHERE state NOT IN ('ADDED_VALID_DOCUMENT','REUSED_EXISTING_DOCUMENT','PIPELINE_FAILURE')
                    OR state IS NULL) FROM outcomes""").fetchone()
                expected = self.scope["audited_coverage"].get("union_native_ids")
                if rows != unique_ids or unfinished or sum(counts.values()) != rows or (expected is not None and rows != expected):
                    raise ValueError(f"Parallel coverage failed: rows={rows}, unique_ids={unique_ids}, unfinished={unfinished}, expected={expected}")
                actual_counts = dict(con.sql("SELECT state,count(*) FROM outcomes GROUP BY 1").fetchall())
                if actual_counts != dict(counts):
                    raise ValueError("Parallel source counters differ from the actual ID outcomes.")
        result = {"state": "FULL_INTAKE_COMPLETE", "completed_sources": len(refs), "total_sources": len(refs),
            "counts": dict(counts), "builds": builds, "audited_coverage": self.scope["audited_coverage"],
            "coverage": {"passed": True, "input_native_ids": rows, "unique_native_ids": unique_ids},
            "scope_sha256": digest_json(self.scope), "parallel_plan_sha256": self.plan_sha,
            "submission_candidate_changed": False, "updated_at": utc_now(),
            "scope": "CPU import/quality/chunks; not embedded, indexed, promoted or submitted"}
        atomic_json(cache, {"identity": identity, "result": result})
        atomic_json(self.root/"full_intake_status.json", result)
        return result

FULL_SCOPE = json.loads('{"schema_version": 1, "audit_date": "2026-10-19", "folder_id": "1P2ke-yIDwj443R_GMegaiH9RACf-Gzth", "base_plan_sha256": "dd4d09120f1e8c0c9c3dc1d977ce34c73e5ac97a58d2df6d56fdc2f13f6fae3f", "base_plan": {"state": "AUDITED_NOT_IMPORTED", "audit_date": "2026-10-19", "active_candidate_action": "KEEP_UNCHANGED", "new_snapshot": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "sources": [{"drive_file_id": "1tVX24iYCthJSYyMsKLdoVBJrgWtUmG-k", "archive": "vibiomir_retain_shard_00001_with_resume_00000.tar", "inner_tar": "vibiomir_shard_00000.tar", "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "frontier": "data/crawl_shards/shard_00000.parquet", "frontier_sha256": "2cf6be6537b6e4c73b25c421503a9e5b7247dddbd10abe8749ae0ac1dbcefcec", "extract_success_url_count_before_pipeline": 11620, "mode": "NEW_SNAPSHOT_SOURCE", "job_id": "new-00000", "archive_bytes": 5262612480}, {"drive_file_id": "1IwebnPLyQhXTphcRtfI7Pg1aZEiXdBEZ", "archive": "vibiomir_retain_shard_00002_with_resume_00001.tar", "inner_tar": "vibiomir_shard_00001.tar", "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "frontier": "data/crawl_shards/shard_00001.parquet", "frontier_sha256": "7b5669ff5fe407205075957696fe846471c2c16cd3a91509406db6caf884c3f0", "extract_success_url_count_before_pipeline": 69498, "mode": "NEW_SNAPSHOT_SOURCE", "job_id": "new-00001", "archive_bytes": 7523225600}, {"drive_file_id": "1IwebnPLyQhXTphcRtfI7Pg1aZEiXdBEZ", "archive": "vibiomir_retain_shard_00002_with_resume_00001.tar", "inner_tar": "vibiomir_shard_00002.tar", "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "frontier": "data/crawl_shards/shard_00002.parquet", "frontier_sha256": "36b93a5a17933403be8de662dd8a985eeb7287025201e056b61970b05f8b7a52", "extract_success_url_count_before_pipeline": 58976, "mode": "NEW_SNAPSHOT_SOURCE", "job_id": "new-00002", "archive_bytes": 7523225600}, {"drive_file_id": "1Z8_0BlP-WwVYkfARLDR7vUtRCa7Xh-69", "archive": "vibiomir_retain_shard_00004_with_resume_00003.tar", "inner_tar": "vibiomir_shard_00003.tar", "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "frontier": "data/crawl_shards/shard_00003.parquet", "frontier_sha256": "cfc373b543d0253978a5fcd27c5f845ce64fbcd8ef349a137ee838ec51d3c2b9", "extract_success_url_count_before_pipeline": 69237, "mode": "NEW_SNAPSHOT_SOURCE", "job_id": "new-00003", "archive_bytes": 7277260800}, {"drive_file_id": "1Z8_0BlP-WwVYkfARLDR7vUtRCa7Xh-69", "archive": "vibiomir_retain_shard_00004_with_resume_00003.tar", "inner_tar": "vibiomir_shard_00004.tar", "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "frontier": "data/crawl_shards/shard_00004.parquet", "frontier_sha256": "18a8db71eefd5b63874e229d42c6411dadd5efbdf810ed01bdc4e977295c2f32", "extract_success_url_count_before_pipeline": 56845, "mode": "NEW_SNAPSHOT_SOURCE", "job_id": "new-00004", "archive_bytes": 7277260800}, {"drive_file_id": "11Yl5P1lFYaCZbcydc5vsUPK2LyJ5fkyC", "archive": "vibiomir_retain_shard_00005.tar", "inner_tar": null, "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "frontier": "data/crawl_shards/shard_00005.parquet", "frontier_sha256": "c102e6c47d672ff031d8eeb227d5491a0f49409bed8b1a24443bed2ea94aa1aa", "extract_success_url_count_before_pipeline": 55323, "mode": "NEW_SNAPSHOT_SOURCE", "job_id": "new-00005", "archive_bytes": 3269468160}], "rules": ["Open nested TAR members explicitly; do not import outer containers as flat archives", "Use exactly one cumulative version of each canonical frontier; preserve original archives", "Bind new shards to their new links snapshot; never relabel them as old-snapshot inputs", "Deduplicate against existing valid documents by exact URL/ID; preserve failed/pending URL ledger entries", "Keep ongoing submission candidate frozen; stage additions without switching active pointers", "EXTRACT_SUCCESS goes through normal quality/chunk gates; only new text hashes need embedding", "Only import sources from the new BTC snapshot; exclude all legacy-snapshot archive jobs"], "schema_version": 1, "campaign": "btc-new-71be0e12-20261019-v1", "tokenizer_revision": "5617a9f61b028005a4858fdac845db406aefb181", "chunking": {"child_tokens": 180, "overlap_tokens": 40, "parent_tokens": 512, "snap_sentences": true, "prefer_sections": true, "dense_max_tokens": 512}, "baseline": {"drive_file_id": "1JBnjci3WUfBlQ0VxKtGeGHTbmhGpydRp", "sha256": "e34ea4e336ea37e42ffb338a3a97986c816b932fdc221c1843d5d3424782ab0c", "candidate_manifest_sha256": "4de4b5c77b867e6017b0f93b56e6d32e1e0c9442ab05d4bdb8e14673e4f7bcda", "documents": 1252445}, "snapshots": {"5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6": {"revision": "71be0e1239a19dfd7a717e2bc13e3bec21bd0334", "filename": "links_corpus_vi_en_zh.parquet", "sha256": "87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7"}}, "required_snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "scope": "NEW_BTC_DATASET_ONLY"}, "additional_sources": [{"drive_file_id": "18cg2bXqfX-6f867we7tdcUEseG52HbZY", "archive": "vibiomir_retain__shard_00006.tar", "archive_bytes": 3489576960, "inner_tar": null, "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "frontier": "data/crawl_shards/shard_00006.parquet", "frontier_sha256": "06823daba41b1be5b6129f961f4a082e14e12e50a4a42e1d6d7335a5f1165296", "extract_success_url_count_before_pipeline": 63806, "job_id": "new-00006", "mode": "NEW_SNAPSHOT_SOURCE"}, {"drive_file_id": "1uA45mDiRXs2Qlb-b0jPqHU9GavA7hHKh", "archive": "vibiomir_retain_shard_00007.tar", "archive_bytes": 3993866240, "inner_tar": null, "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "frontier": "data/crawl_shards/shard_00007.parquet", "frontier_sha256": "e9642c5ab759b21ea320793409c8675809af4d8d171562a6ffe9acfcde444800", "extract_success_url_count_before_pipeline": 69940, "job_id": "new-00007", "mode": "NEW_SNAPSHOT_SOURCE"}], "retained": {"job_id": "retained-71be0e12", "mode": "NEW_SNAPSHOT_SOURCE", "format": "RETAINED_PORTABLE_V1", "archive": "vibiomir_retained_71be0e12.tar", "drive_file_id": "1Hf6cb7Des8AhYkyWXE3d3PvnDT-BMMQ-", "archive_bytes": 12581928960, "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "original_snapshot_id": "b915d5d5d640d497d103fa37a5a047c1d093772ab17b09638634de4bbc80e5d7", "official_links_sha256": "87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7", "members": {"mapping": {"path": "data/doc_url_map.parquet", "offset": 12562358272, "bytes": 19545473, "sha256": "1dc06041fded26cfe2dfa0c64e39875840a8b3a8051ac58080608497fadaee01"}, "raw_catalog": {"path": "data/raw_catalog.parquet", "offset": 11313683456, "bytes": 125719140, "sha256": "90e5ae894488dd7e47a76dd4664bcaa8156df72245f18c078db1b66ac0f4f9e9"}, "documents": {"path": "data/documents.parquet", "offset": 11439404544, "bytes": 1122951853, "sha256": "b031ec37043f1de3e6f24a70e508fd3e569fb8dff319a59c69489a3bd8adabc3"}, "dataset": {"path": "new_dataset_manifest.json", "offset": 12581909504, "bytes": 1093, "sha256": "912ac3c910985b5f3f003f3d90ea317b13c9c568da7baf424d87488b1cfe7d31"}}}, "required_physical_archive_names": ["vibiomir_retain_shard_00001_with_resume_00000.tar", "vibiomir_retain_shard_00002_with_resume_00001.tar", "vibiomir_retain_shard_00004_with_resume_00003.tar", "vibiomir_retain_shard_00005.tar", "vibiomir_retain__shard_00006.tar", "vibiomir_retain_shard_00007.tar", "vibiomir_retained_71be0e12.tar"], "duplicate_versions_to_skip": ["vibiomir_retain_shard_00000.tar", "vibiomir_retain_shard_00003.tar", "inner vibiomir_shard_00001.tar in vibiomir_retain_shard_00001_with_resume_00000.tar"], "audited_coverage": {"frontier_urls": 800000, "frontier_native_ids": 800999, "retained_urls": 316097, "retained_native_ids": 316693, "union_native_ids": 1117692, "official_native_ids": 1220598, "note": "Input membership coverage, not successful extraction or completed pipeline coverage"}}')

import json
from huggingface_hub import hf_hub_download
from vietmedbridge.artifacts import verify_file, publish_file, digest_json
from vietmedbridge.isolated_intake import DriveSource, fetch_reference
from vietmedbridge.chunks import load_bge_tokenizer, ChunkConfig
from vietmedbridge.golden import run_golden_suite

def validate_new_btc_plan(plan):
    required = "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6"
    if (plan.get("required_snapshot_id") != required
            or set(plan["snapshots"]) != {required}
            or not plan["sources"]
            or any(source["snapshot_id"] != required
                   or source["mode"] != "NEW_SNAPSHOT_SOURCE"
                   for source in plan["sources"])):
        raise ValueError("Lượt này chỉ nhận URL thuộc snapshot BTC mới 71be0e12; không nhập nguồn legacy.")

PLAN = FULL_SCOPE["base_plan"]
validate_new_btc_plan(PLAN)
CONFIG = ChunkConfig(**PLAN["chunking"])
TOKENIZER_SPEC = {"model_id": "BAAI/bge-m3", "revision": PLAN["tokenizer_revision"], "special_tokens": False}
CONTROLLER_OPTIONS = dict(root=INTAKE_ROOT, work_dir=WORK_DIR,
    tokenizer_spec=TOKENIZER_SPEC, chunk_config=CONFIG, shard_size=2048,
    controller_type=IntakeResume, retained_type=RetainedAdapter,
    sql_context=bounded_intake_sql, parquet_writer=bounded_intake_parquet)
if RUN_MODE == "sequential":
    CONTROLLER = FullIntake(FULL_SCOPE, **CONTROLLER_OPTIONS)
elif RUN_MODE in ("parallel", "collect"):
    CONTROLLER = ParallelIntake(FULL_SCOPE, worker_id=WORKER_ID, team_size=TEAM_SIZE, **CONTROLLER_OPTIONS)
else:
    raise ValueError("RUN_MODE phải là sequential, parallel hoặc collect.")
REFERENCE_DIR = (INTAKE_ROOT / "references" if RUN_MODE == "sequential" else
                 INTAKE_ROOT / "parallel_cpu4" / f"worker-{WORKER_ID}" / "references")
CHECKPOINT_STATUS = CONTROLLER.status()
print("Checkpoint:", CHECKPOINT_STATUS["completed_sources"], "/", CHECKPOINT_STATUS["total_sources"],
      "nguồn đã xong | chỉ đọc receipt nhỏ, chưa tải lại dữ liệu cũ", flush=True)
LAZY_RESOURCES = {}

def load_reader():
    if "reader" not in LAZY_RESOURCES:
        from google.colab import auth
        import google.auth
        from google.auth.transport.requests import AuthorizedSession
        auth.authenticate_user()
        credentials, _ = google.auth.default()
        LAZY_RESOURCES["reader"] = DriveSource(AuthorizedSession(credentials))
    return LAZY_RESOURCES["reader"]

def load_baseline():
    if "baseline" not in LAZY_RESOURCES:
        reference = PLAN["baseline"]
        shared = INTAKE_ROOT / "references/baseline_aliases.parquet"
        target = shared if shared.is_file() else REFERENCE_DIR / "baseline_aliases.parquet"
        if not target.is_file():
            fetch_reference(load_reader(), reference, target)
        print("Cần mapping cũ để chọn phần mới; chỉ đưa mapping lên ổ local một lần.", flush=True)
        LAZY_RESOURCES["baseline"] = intake_core._local_mapping(target, reference["sha256"], WORK_DIR)
    return LAZY_RESOURCES["baseline"]

def load_links(snapshot_id):
    key = "links-"+snapshot_id
    if key not in LAZY_RESOURCES:
        spec = PLAN["snapshots"][snapshot_id]
        shared = INTAKE_ROOT / "references" / (snapshot_id[:16]+"-links.parquet")
        target = shared if shared.is_file() else REFERENCE_DIR / (snapshot_id[:16]+"-links.parquet")
        if not target.is_file():
            print("Tải official links:", spec["revision"], flush=True)
            downloaded = Path(hf_hub_download("AIGuruTinix/ViBioMIR", spec["filename"],
                repo_type="dataset", revision=spec["revision"]))
            verify_file(downloaded, spec["sha256"])
            publish_file(downloaded, target)
        LAZY_RESOURCES[key] = intake_core._local_mapping(target, spec["sha256"], WORK_DIR)
    return LAZY_RESOURCES[key]

def load_tokenizer():
    if "tokenizer" not in LAZY_RESOURCES:
        tokenizer, spec = load_bge_tokenizer(PLAN["tokenizer_revision"])
        golden = run_golden_suite(CHECKOUT / "tests/golden/cases.json", tokenizer, spec, CONFIG)
        atomic_json((INTAKE_ROOT if RUN_MODE == "sequential" else REFERENCE_DIR.parent) / "golden.json", golden)
        if not golden["passed"]:
            raise RuntimeError("Golden failed; chưa chia chunk mới.")
        LAZY_RESOURCES["tokenizer"] = (tokenizer, spec)
    return LAZY_RESOURCES["tokenizer"]


In [ ]:
#@title 3. Chỉ tiếp tục nguồn thiếu, lưu checkpoint từng shard và từng nguồn

def continue_intake():
    if RUN_MODE == "collect":
        return CONTROLLER.collect()
    return CONTROLLER.run(load_reader=load_reader, load_links=load_links,
        load_baseline=load_baseline, load_tokenizer=load_tokenizer)

try:
    RESULT = continue_intake()
except OSError as error:
    if error.errno not in (103, 107):
        raise
    print("Drive ngắt kết nối; gắn lại một lần rồi tiếp tục checkpoint intake.", flush=True)
    drive.mount("/content/drive", force_remount=True)
    RESULT = continue_intake()
print(json.dumps(RESULT, ensure_ascii=False, indent=2))
if RUN_MODE == "parallel":
    print("Worker hoàn tất phần được giao; chưa đồng nghĩa 9/9 nguồn. Acc 0 chạy cell 4 để tổng hợp.")
else:
    print("Bàn giao toàn bộ lượt data mới:", INTAKE_ROOT / "full_intake_status.json")
print("Giữ nguyên luồng submission đang chạy.")


In [ ]:
#@title 4. Tổng hợp 4 CPU workers (acc 0; chạy lại cell này sau khi cả 4 xong)
if RUN_MODE == "parallel" and WORKER_ID == 0:
    COLLECTION = CONTROLLER.collect()
    print(json.dumps(COLLECTION, ensure_ascii=False, indent=2))
    if COLLECTION["state"] == "WAITING_FOR_CPU_WORKERS":
        print("Chờ các nguồn trong pending_sources. Khi các acc xong, chỉ chạy lại cell 4 này.")
elif RUN_MODE == "parallel":
    print(f"Acc worker {WORKER_ID} đã lưu phần được giao. Acc 0 tổng hợp bằng cell 4.")


`FULL_INTAKE_COMPLETE` và `completed_sources=9/9` nghĩa là mọi ID trong
toàn bộ nguồn đã audit có kết quả: `ADDED_VALID_DOCUMENT`,
`REUSED_EXISTING_DOCUMENT` hoặc `PIPELINE_FAILURE`.
`INTAKE_COMPLETE` chỉ xác nhận một chặng trong lượt đầy đủ.
URL lỗi được giữ trong `sources/<job>/source_outcomes.parquet` và failure
shards; trạng thái này không có nghĩa là crawl thành công 100%.

`SKIP_COMPLETED_SOURCE`: bỏ qua nguồn xong, chỉ đọc receipt/trạng thái file.
`REUSE_SELECTION`: tiếp tục nguồn dở, không chọn lại URL hoặc nạp mapping cũ.
`RESUME_PARTS`: giữ shard đã commit; chỉ source đang dở cần kiểm tra checksum.
`REUSE_VALIDATED_BUILD`: build đã qua integrity, chỉ hoàn tất bước bàn giao.
Mapping được kiểm SHA khi thực sự sử dụng; receipt không thay thế
một lần audit lại toàn bộ payload. Không nạp catalog/BM25 hoặc embedding cũ.

Gói retained được kiểm mapping BTC mới và SHA nguồn cũ, không đổi snapshot
của lần crawl/extraction. Nhóm LOW/QUARANTINE giữ để review. Metadata lớn
tải theo range và lưu checkpoint mỗi 8 MiB; không tải raw HTML của TAR.

Gửi `full_intake_status.json` sau khi xong để kiểm tra số thực tế. Bước kế tiếp
là freeze/chuẩn bị index cho **lượt thêm riêng**, rồi chỉ embedding text
chưa có; ghép vào phiên submission kế tiếp sau khi ZIP hiện tại hoàn tất.
